<a href="https://colab.research.google.com/github/XCXAA/PyTorch-Tutorial/blob/main/optimization_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\| **Optimization** \|\| [Save &
Load Model](saveloadrun_tutorial.html)

Optimizing Model Parameters
===========================

Now that we have a model and data it\'s time to train, validate and test
our model by optimizing its parameters on our data. Training a model is
an iterative process; in each iteration the model makes a guess about
the output, calculates the error in its guess (*loss*), collects the
derivatives of the error with respect to its parameters (as we saw in
the [previous section](autogradqs_tutorial.html)), and **optimizes**
these parameters using gradient descent. For a more detailed walkthrough
of this process, check out this video on [backpropagation from
3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8).

Prerequisite Code
-----------------

We load the code from the previous sections on [Datasets &
DataLoaders](data_tutorial.html) and [Build
Model](buildmodel_tutorial.html).


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST( #Tells PyTorch that you want to use the Fashion-MNIST dataset.
    root="data",  #This tells PyTorch where to store the dataset on your computer.
    train=True,  # give me the training portion of Fashion-MNIST.
    download=True,  # If the dataset isn't already in the data folder, PyTorch will download it automatically.
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
) #Converts the downloaded image into a PyTorch image representation (torch.Tensor/image tensor format).
  # Converts the pixel values to floating-point numbers and Scales the pixel values from 0–255 to 0–1.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64) #Instead of giving all the example at once, we split them into batches, in this case we have 64 batches
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__() #This initializes the parent class, nn.Module.
        self.flatten = nn.Flatten() #Flatten() changes 28 x 28 to 784 x 1
        self.linear_relu_stack = nn.Sequential( #"Run these layers one after another in this exact order."
            nn.Linear(28*28, 512), #It takes 784 input numbers and produces 512 numbers.
            nn.ReLU(),  #Activation function: =max(0,x)
            nn.Linear(512, 512),  #This takes the 512 values from the previous layer and produces another 512 values.
            nn.ReLU(),
            nn.Linear(512, 10), #the final layer produces 10 numbers, since the Fashion-MNIST has 10 different classes
        )

    def forward(self, x): #forward() describes how data travels through the network.
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()
"""
Fashion-MNIST image
       │
       │ 28 × 28
       ↓
   Flatten()
       │
       │ 784
       ↓
 Linear(784 → 512)
       │
       ↓
     ReLU
       │
       │ 512
       ↓
 Linear(512 → 512)
       │
       ↓
     ReLU
       │
       │ 512
       ↓
 Linear(512 → 10)
       │
       ↓
   10 logits
       │
       ↓
Prediction
"""

100%|██████████| 26.4M/26.4M [00:03<00:00, 8.69MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 143kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.63MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 18.4MB/s]


Hyperparameters
===============

Hyperparameters are adjustable parameters that let you control the model
optimization process. Different hyperparameter values can impact model
training and convergence rates ([read
more](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)
about hyperparameter tuning)

We define the following hyperparameters for training:

-   **Number of Epochs** - the number of times to iterate over the
    dataset
-   **Batch Size** - the number of data samples propagated through the
    network before the parameters are updated
-   **Learning Rate** - how much to update models parameters at each
    batch/epoch. Smaller values yield slow learning speed, while large
    values may result in unpredictable behavior during training.


In [2]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

Optimization Loop
=================

Once we set our hyperparameters, we can then train and optimize our
model with an optimization loop. Each iteration of the optimization loop
is called an **epoch**.

Each epoch consists of two main parts:

-   **The Train Loop** - iterate over the training dataset and try to
    converge to optimal parameters.
-   **The Validation/Test Loop** - iterate over the test dataset to
    check if model performance is improving.

Let\'s briefly familiarize ourselves with some of the concepts used in
the training loop. Jump ahead to see the
`full-impl-label`{.interpreted-text role="ref"} of the optimization
loop.

Loss Function
-------------

When presented with some training data, our untrained network is likely
not to give the correct answer. **Loss function** measures the degree of
dissimilarity of obtained result to the target value, and it is the loss
function that we want to minimize during training. To calculate the loss
we make a prediction using the inputs of our given data sample and
compare it against the true data label value.

Common loss functions include
[nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)
(Mean Square Error) for regression tasks, and
[nn.NLLLoss](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss)
(Negative Log Likelihood) for classification.
[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss)
combines `nn.LogSoftmax` and `nn.NLLLoss`.

We pass our model\'s output logits to `nn.CrossEntropyLoss`, which will
normalize the logits and compute the prediction error.


In [3]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

Optimizer
=========

Optimization is the process of adjusting model parameters to reduce
model error in each training step. **Optimization algorithms** define
how this process is performed (in this example we use Stochastic
Gradient Descent). All optimization logic is encapsulated in the
`optimizer` object. Here, we use the SGD optimizer; additionally, there
are many [different
optimizers](https://pytorch.org/docs/stable/optim.html) available in
PyTorch such as ADAM and RMSProp, that work better for different kinds
of models and data.

We initialize the optimizer by registering the model\'s parameters that
need to be trained, and passing in the learning rate hyperparameter.


In [4]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # Instead of calculating the gradient using the entire training dataset, SGD typically uses a small batch of examples.

Inside the training loop, optimization happens in three steps:

-   Call `optimizer.zero_grad()` to reset the gradients of model
    parameters. Gradients by default add up; to prevent double-counting,
    we explicitly zero them at each iteration.
-   Backpropagate the prediction loss with a call to `loss.backward()`.
    PyTorch deposits the gradients of the loss w.r.t. each parameter.
-   Once we have our gradients, we call `optimizer.step()` to adjust the
    parameters by the gradients collected in the backward pass.


Full Implementation {#full-impl-label}
===================

We define `train_loop` that loops over our optimization code, and
`test_loop` that evaluates the model\'s performance against our test
data.


In [5]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]") #Current Training Error, total images, total inputs so far


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset) #This tells you how many images are in the test dataset.
    num_batches = len(dataloader) #Remember that the dataloader gives the images in batches.
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:  #X = images,y = correct answers
            pred = model(X)
            test_loss += loss_fn(pred, y).item() #Converts the PyTorch tensor containing the loss into a normal Python number.
            correct += (pred.argmax(1) == y).type(torch.float).sum().item() #type(torch.float) converts True/False to number 1 or 0 and then sums

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

We initialize the loss function and optimizer, and pass it to
`train_loop` and `test_loop`. Feel free to increase the number of epochs
to track the model\'s improving performance.


In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.315784  [   64/60000]
loss: 2.293420  [ 6464/60000]
loss: 2.280142  [12864/60000]
loss: 2.267502  [19264/60000]
loss: 2.261519  [25664/60000]
loss: 2.230065  [32064/60000]
loss: 2.243795  [38464/60000]
loss: 2.212264  [44864/60000]
loss: 2.213457  [51264/60000]
loss: 2.164607  [57664/60000]
Test Error: 
 Accuracy: 33.6%, Avg loss: 2.166924 

Epoch 2
-------------------------------
loss: 2.190070  [   64/60000]
loss: 2.168016  [ 6464/60000]
loss: 2.119105  [12864/60000]
loss: 2.126183  [19264/60000]
loss: 2.090257  [25664/60000]
loss: 2.033684  [32064/60000]
loss: 2.067399  [38464/60000]
loss: 1.992938  [44864/60000]
loss: 2.005277  [51264/60000]
loss: 1.923671  [57664/60000]
Test Error: 
 Accuracy: 48.4%, Avg loss: 1.918346 

Epoch 3
-------------------------------
loss: 1.966684  [   64/60000]
loss: 1.922735  [ 6464/60000]
loss: 1.814807  [12864/60000]
loss: 1.842083  [19264/60000]
loss: 1.755869  [25664/60000]
loss: 1.702528  [32064/600

Further Reading
===============

-   [Loss
    Functions](https://pytorch.org/docs/stable/nn.html#loss-functions)
-   [torch.optim](https://pytorch.org/docs/stable/optim.html)
-   [Warmstart Training a
    Model](https://pytorch.org/tutorials/recipes/recipes/warmstarting_model_using_parameters_from_a_different_model.html)
